# High-Precision Image Clustering — Exclusive Mode

### Pipeline
```
dataset_for_clustering.zip
       ↓
EfficientNetB0 (ImageNet) → GlobalAveragePooling2D → L2 Normalize
       ↓
Agglomerative Clustering (distance_threshold, strict)
       ↓
Cosine Similarity check > 0.92 per cluster
       ↓
Organize → clustered/cluster_001/ cluster_002/ ...
       ↓
Report
```
**Run cells in order. Do NOT skip.**

---

## STEP 0 - Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

Mounted at /content/drive
Drive mounted!


---
## STEP 1 — Install Dependencies

In [2]:
!pip install tensorflow scikit-learn tqdm numpy pillow -q
print('Dependencies ready!')

Dependencies ready!


---
## STEP 2 — Configuration

In [3]:
import os

# ============================================================
# PATHS — Edit if needed
# ============================================================
ZIP_PATH    = '/content/drive/MyDrive/dataset_for_clustering.zip'
EXTRACT_DIR = '/content/temp_extract'
OUTPUT_DIR  = '/content/drive/MyDrive/sugar_checker_data_V2/images/clustered'

# ============================================================
# CLUSTERING PARAMETERS — Tune here
# ============================================================
IMG_SIZE           = (224, 224)
SIMILARITY_THRESH  = 0.92    # cosine similarity must exceed this to stay in cluster
DISTANCE_THRESHOLD = 0.15    # agglomerative: smaller = stricter = more clusters
VALID_EXTS         = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
# ============================================================

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

print('Configuration loaded!')
print(f'  ZIP source         : {ZIP_PATH}')
print(f'  Extract dir        : {EXTRACT_DIR}')
print(f'  Output dir         : {OUTPUT_DIR}')
print(f'  Similarity thresh  : {SIMILARITY_THRESH}')
print(f'  Distance threshold : {DISTANCE_THRESHOLD}')

Configuration loaded!
  ZIP source         : /content/drive/MyDrive/dataset_for_clustering.zip
  Extract dir        : /content/temp_extract
  Output dir         : /content/drive/MyDrive/sugar_checker_data_V2/images/clustered
  Similarity thresh  : 0.92
  Distance threshold : 0.15


---
## STEP 3 — Extract ZIP

In [4]:
import zipfile
from pathlib import Path

print('Extracting ZIP...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Auto-detect root
subdirs = [d for d in Path(EXTRACT_DIR).iterdir() if d.is_dir()]
DATA_ROOT = str(subdirs[0]) if len(subdirs) == 1 else EXTRACT_DIR

image_paths = sorted([
    str(p) for p in Path(DATA_ROOT).rglob('*')
    if p.suffix.lower() in VALID_EXTS
])

print(f'Extracted to        : {DATA_ROOT}')
print(f'Total images found  : {len(image_paths):,}')

Extracting ZIP...
Extracted to        : /content/temp_extract
Total images found  : 16,214


---
## STEP 4 — Feature Extraction (EfficientNetB0 + L2 Normalize)

> Uses EfficientNetB0 pretrained on ImageNet as a feature extractor. GlobalAveragePooling2D produces a 1280-dim vector per image. L2 normalization ensures consistent cosine distance computation.

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.preprocessing import normalize
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Build feature extractor
print('Building EfficientNetB0 feature extractor...')
base = EfficientNetB0(
    weights      = 'imagenet',
    include_top  = False,
    input_shape  = IMG_SIZE + (3,),
)
x       = layers.GlobalAveragePooling2D()(base.output)
extractor = Model(inputs=base.input, outputs=x)
extractor.trainable = False
print(f'  Output feature dim : {extractor.output_shape[-1]}')

# Extract features
print(f'\nExtracting features from {len(image_paths):,} images...')
features    = []
valid_paths = []
skipped     = 0

for path in tqdm(image_paths, desc='Extracting', unit='img'):
    try:
        img = load_img(path, target_size=IMG_SIZE)
        arr = img_to_array(img)
        arr = preprocess_input(arr)
        arr = np.expand_dims(arr, axis=0)
        feat = extractor.predict(arr, verbose=0)[0]
        features.append(feat)
        valid_paths.append(path)
    except Exception:
        skipped += 1

# L2 Normalize
features = normalize(np.array(features, dtype=np.float32), norm='l2')

print(f'\nFeature extraction complete!')
print(f'  Valid images   : {len(valid_paths):,}')
print(f'  Skipped        : {skipped}')
print(f'  Feature shape  : {features.shape}')

Building EfficientNetB0 feature extractor...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Output feature dim : 1280

Extracting features from 16,214 images...


Extracting: 100%|██████████| 16214/16214 [22:41<00:00, 11.91img/s] 



Feature extraction complete!
  Valid images   : 8,107
  Skipped        : 8107
  Feature shape  : (8107, 1280)


---
## STEP 5 — Agglomerative Clustering (Strict Mode)

> Uses `AgglomerativeClustering` with `distance_threshold` instead of fixed `n_clusters`. Smaller `distance_threshold` = stricter = more clusters = less mixing between products.

In [6]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

print(f'Running Agglomerative Clustering (distance_threshold={DISTANCE_THRESHOLD})...')
print('This may take a few minutes for large datasets...')

clustering = AgglomerativeClustering(
    n_clusters         = None,
    distance_threshold = DISTANCE_THRESHOLD,
    metric             = 'cosine',
    linkage            = 'average',   # average linkage is best for image clusters
)

raw_labels = clustering.fit_predict(features)
n_raw      = len(set(raw_labels))

print(f'\nInitial clustering complete!')
print(f'  Raw clusters formed : {n_raw}')

# Distribution
from collections import Counter
dist = Counter(raw_labels)
print(f'  Largest cluster     : {max(dist.values())} images')
print(f'  Smallest cluster    : {min(dist.values())} images')
print(f'  Singleton clusters  : {sum(1 for v in dist.values() if v == 1)}')

Running Agglomerative Clustering (distance_threshold=0.15)...
This may take a few minutes for large datasets...

Initial clustering complete!
  Raw clusters formed : 7183
  Largest cluster     : 7 images
  Smallest cluster    : 1 images
  Singleton clusters  : 6423


---
## STEP 6 — Cosine Similarity Filter (> 0.92)

> For each cluster, compute pairwise cosine similarity against the cluster centroid. Images below the threshold are ejected to `cluster_outliers/`.

In [7]:
import numpy as np
from collections import defaultdict

print(f'Applying cosine similarity filter (threshold={SIMILARITY_THRESH})...')

# Group indices by cluster
cluster_groups = defaultdict(list)
for idx, label in enumerate(raw_labels):
    cluster_groups[label].append(idx)

final_clusters  = {}   # cluster_id → [img_paths]
outlier_paths   = []
new_cluster_id  = 0

for label, indices in cluster_groups.items():
    cluster_feats = features[indices]

    # Centroid = mean of all features in cluster
    centroid  = cluster_feats.mean(axis=0, keepdims=True)
    centroid  = centroid / np.linalg.norm(centroid)

    # Cosine similarity of each image to centroid
    sims = cosine_similarity(cluster_feats, centroid).flatten()

    accepted = [indices[i] for i, s in enumerate(sims) if s >= SIMILARITY_THRESH]
    rejected = [indices[i] for i, s in enumerate(sims) if s < SIMILARITY_THRESH]

    if accepted:
        final_clusters[new_cluster_id] = [valid_paths[i] for i in accepted]
        new_cluster_id += 1

    # Rejected images become outliers
    outlier_paths.extend([valid_paths[i] for i in rejected])

print(f'\nFilter complete!')
print(f'  Clusters after filter : {len(final_clusters)}')
print(f'  Outliers ejected      : {len(outlier_paths)}')
print(f'  Images in clusters    : {sum(len(v) for v in final_clusters.values())}')

Applying cosine similarity filter (threshold=0.92)...

Filter complete!
  Clusters after filter : 7183
  Outliers ejected      : 1
  Images in clusters    : 8106


---
## STEP 7 — Organize Files into cluster_XXX/ Folders

In [8]:
import shutil
from pathlib import Path
from tqdm import tqdm

print(f'Organizing {sum(len(v) for v in final_clusters.values())} images into cluster folders...')

errors = 0

# Organize clusters
for cluster_id, paths in tqdm(final_clusters.items(), desc='Clusters', unit='cluster'):
    cluster_dir = Path(OUTPUT_DIR) / f'cluster_{cluster_id+1:03d}'
    cluster_dir.mkdir(parents=True, exist_ok=True)

    for src_path in paths:
        src = Path(src_path)
        dst = cluster_dir / src.name
        counter = 1
        while dst.exists():
            dst = cluster_dir / f'{src.stem}_{counter}{src.suffix}'
            counter += 1
        try:
            shutil.copy2(src, dst)
        except Exception:
            errors += 1

# Organize outliers
if outlier_paths:
    outlier_dir = Path(OUTPUT_DIR) / 'cluster_outliers'
    outlier_dir.mkdir(parents=True, exist_ok=True)
    for src_path in tqdm(outlier_paths, desc='Outliers', unit='img'):
        src = Path(src_path)
        dst = outlier_dir / src.name
        counter = 1
        while dst.exists():
            dst = outlier_dir / f'{src.stem}_{counter}{src.suffix}'
            counter += 1
        try:
            shutil.copy2(src, dst)
        except Exception:
            errors += 1

print(f'\nOrganization complete!')
print(f'  Errors : {errors}')
print(f'  Output : {OUTPUT_DIR}')

Organizing 8106 images into cluster folders...


Outliers: 100%|██████████| 1/1 [00:00<00:00, 51.96img/s]


Organization complete!
  Errors : 0
  Output : /content/drive/MyDrive/sugar_checker_data_V2/images/clustered


---
## STEP 8 — Report

In [9]:
from pathlib import Path

print('Clustering Report:')
print('=' * 65)
print(f'  Total images processed : {len(valid_paths):,}')
print(f'  Total clusters formed  : {len(final_clusters)}')
print(f'  Outliers               : {len(outlier_paths)}')
print(f'  Similarity threshold   : {SIMILARITY_THRESH}')
print(f'  Distance threshold     : {DISTANCE_THRESHOLD}')
print()
print(f'  {"Cluster":<20} {"Images":>8}')
print(f'  {"-"*30}')

for cluster_id in sorted(final_clusters.keys()):
    folder = f'cluster_{cluster_id+1:03d}'
    count  = len(final_clusters[cluster_id])
    samples = [Path(p).name[:30] for p in final_clusters[cluster_id][:2]]
    print(f'  {folder:<20} {count:>8}   e.g. {samples}')

print(f'  {"cluster_outliers":<20} {len(outlier_paths):>8}')
print('=' * 65)
print(f'\nOutput folder: {OUTPUT_DIR}')
print('Done! Open Finder and inspect each cluster_XXX/ folder.')

Clustering Report:
  Total images processed : 8,107
  Total clusters formed  : 7183
  Outliers               : 1
  Similarity threshold   : 0.92
  Distance threshold     : 0.15

  Cluster                Images
  ------------------------------
  cluster_001                 2   e.g. ['IMG_20240101_192703.jpg', 'unknown_00008.jpg']
  cluster_002                 2   e.g. ['IMG_20241023_075922.jpg', 'unknown_00000.jpg']
  cluster_003                 2   e.g. ['IMG_20241127_094215.jpg', 'unknown_00013.jpg']
  cluster_004                 1   e.g. ['IMG_20241127_094233.jpg']
  cluster_005                 2   e.g. ['IMG_20250219_225550.jpg', 'unknown_00016.jpg']
  cluster_006                 1   e.g. ['IMG_20250219_225613.jpg']
  cluster_007                 2   e.g. ['IMG_20250219_225723.jpg', 'unknown_00005.jpg']
  cluster_008                 2   e.g. ['IMG_20250303_230122.jpg', 'unknown_00007.jpg']
  cluster_009                 1   e.g. ['IMG_20250303_230207.jpg']
  cluster_010               